# 🔍 Multimodal Fake Product Hype Detection
## Step 1 — Project Setup, Folder Structure & Environment Validation

**Goal:** Verify your environment is fully configured, create all project directories,  
load the central config, and confirm all packages are installed correctly.

> **After completing this notebook, say `NEXT` to proceed to Step 2: Dataset Loading.**

---
### 📌 What this notebook covers
| # | Section | Purpose |
|---|---------|---------|
| 1 | Project Directory Setup | Create all folders programmatically |
| 2 | Configuration | Load and display `config.yaml` |
| 3 | Environment Validation | Check Python version and packages |
| 4 | NLTK / spaCy Resources | Download NLP corpora |
| 5 | Reproducibility | Fix all random seeds |
| 6 | Quick Sanity Test | Smoke-test key imports |
| 7 | Summary | Print final health report |

## Section 1 — Project Directory Setup

We create the entire folder tree programmatically so the project is **self-contained**  
and reproducible on any machine. Each directory has a clear, single responsibility.

```
ML Project/
├── data/           ← datasets (raw → interim → processed → features)
├── models/         ← saved model artifacts
├── notebooks/      ← step-by-step Jupyter notebooks
├── src/            ← all Python source code modules
├── reports/        ← figures, tables, visualizations
├── paper/          ← research paper draft
├── scripts/        ← CLI utilities and setup scripts
├── tests/          ← unit tests
├── config/         ← YAML configuration
└── logs/           ← runtime logs
```

In [ ]:
"""
Cell 1: Create all project directories.
Run this once when first setting up the project.
"""

import os
import sys
from pathlib import Path

# ── Ensure we always run from the project root ───────────────
# If the notebook is inside 'notebooks/', go up one level.
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

# ── Define the full directory taxonomy ───────────────────────
DIRECTORIES = [
    # Data pipeline layers
    "data/raw",
    "data/interim",
    "data/processed",
    "data/features",
    # Model artifacts
    "models/saved",
    "models/checkpoints",
    "models/hf_cache",
    # Reports
    "reports/figures",
    "reports/tables",
    # Source modules
    "src/data",
    "src/preprocessing",
    "src/features",
    "src/models",
    "src/evaluation",
    "src/visualization",
    "src/utils",
    # Misc
    "notebooks",
    "paper",
    "scripts",
    "tests",
    "logs",
    "config",
]

created = []
already_exists = []

for d in DIRECTORIES:
    path = Path(d)
    if not path.exists():
        path.mkdir(parents=True, exist_ok=True)
        # Add a .gitkeep so the empty dir is tracked by Git
        (path / ".gitkeep").touch()
        created.append(d)
    else:
        already_exists.append(d)

print(f"\n✔ Created  : {len(created)} new directories")
print(f"✔ Existing : {len(already_exists)} directories already present")
print("\nFull directory tree:")
for d in DIRECTORIES:
    status = "NEW" if d in created else "OK "
    print(f"  [{status}] {d}/")

## Section 2 — Configuration

The `config/config.yaml` file is the **single source of truth** for:
- File paths
- Model hyperparameters  
- Feature engineering settings
- Hype score weights and thresholds

We load it using the `src/utils/config_loader.py` helper, which converts the YAML  
into a dot-accessible Python object (e.g., `cfg.models.xgboost.n_estimators`).

In [ ]:
"""
Cell 2: Load and display the project config.
"""

import yaml
import json
from types import SimpleNamespace

def _dict_to_namespace(d):
    """Recursively convert dict → SimpleNamespace for dot-access."""
    ns = SimpleNamespace()
    for k, v in d.items():
        setattr(ns, k, _dict_to_namespace(v) if isinstance(v, dict) else v)
    return ns

def load_config(path="config/config.yaml"):
    """Load YAML config and return dot-accessible namespace."""
    config_path = Path(path)
    if not config_path.exists():
        print(f"⚠  Config not found at {config_path.resolve()}")
        print("   Make sure config/config.yaml exists in the project root.")
        return None
    with open(config_path, "r") as f:
        raw = yaml.safe_load(f)
    return _dict_to_namespace(raw)

# ── Load Config ───────────────────────────────────────────────
cfg = load_config()

if cfg is not None:
    print("=" * 55)
    print(f"  Project : {cfg.project.name}")
    print(f"  Version : {cfg.project.version}")
    print(f"  Seed    : {cfg.project.seed}")
    print("=" * 55)
    print(f"\n  Paths:")
    print(f"    Raw data    : {cfg.paths.raw_data}")
    print(f"    Processed   : {cfg.paths.processed_data}")
    print(f"    Features    : {cfg.paths.features}")
    print(f"    Models      : {cfg.paths.models}")
    print(f"\n  Hype Score Weights:")
    print(f"    Fake review probability : {cfg.hype_score.weights.fake_review_prob}")
    print(f"    Temporal anomaly        : {cfg.hype_score.weights.temporal_anomaly}")
    print(f"    Reviewer behavior       : {cfg.hype_score.weights.reviewer_behavior}")
    print(f"    Rating variance         : {cfg.hype_score.weights.rating_variance}")
    print(f"\n  Risk Thresholds:")
    print(f"    Low    : 0  – {cfg.hype_score.thresholds.low}")
    print(f"    Medium : {cfg.hype_score.thresholds.low+1} – {cfg.hype_score.thresholds.medium}")
    print(f"    High   : {cfg.hype_score.thresholds.medium+1} – 100")
else:
    print("Config not loaded. Please check the config/config.yaml file.")

## Section 3 — Environment Validation

We check:
1. **Python version** — must be ≥ 3.9
2. **Core packages** — all required libraries installed with correct versions
3. **GPU availability** — optional but beneficial for BERT fine-tuning

Missing packages: run `pip install -r requirements.txt`

In [ ]:
"""
Cell 3: Environment validation — Python version + package check.
"""

import sys
import importlib
import platform

# ── Python Version Check ──────────────────────────────────────
ver = sys.version_info
py_ok = (ver.major == 3 and ver.minor >= 9)
status_icon = "✔" if py_ok else "✘"
print(f"{status_icon}  Python {ver.major}.{ver.minor}.{ver.micro} "
      f"({'OK' if py_ok else 'UPGRADE REQUIRED — need 3.9+'})")
print(f"   Platform : {platform.system()} {platform.machine()}")
print(f"   Executable: {sys.executable}\n")

# ── Package Registry ───────────────────────────────────────────
# (import_name, display_name, minimum_version)
PACKAGES = [
    ("numpy",            "NumPy",              "1.24"),
    ("pandas",           "Pandas",             "2.0"),
    ("sklearn",          "Scikit-learn",       "1.3"),
    ("scipy",            "SciPy",              "1.10"),
    ("xgboost",          "XGBoost",            "1.7"),
    ("lightgbm",         "LightGBM",           "4.0"),
    ("torch",            "PyTorch",            "2.0"),
    ("transformers",     "HuggingFace Transformers", "4.30"),
    ("sentence_transformers", "Sentence-Transformers", "2.2"),
    ("nltk",             "NLTK",               "3.8"),
    ("spacy",            "spaCy",              "3.5"),
    ("textblob",         "TextBlob",           "0.17"),
    ("vaderSentiment",   "VADER Sentiment",    "3.3"),
    ("shap",             "SHAP",               "0.41"),
    ("matplotlib",       "Matplotlib",         "3.7"),
    ("seaborn",          "Seaborn",            "0.12"),
    ("plotly",           "Plotly",             "5.14"),
    ("tqdm",             "tqdm",               "4.64"),
    ("yaml",             "PyYAML",             "6.0"),
    ("loguru",           "Loguru",             "0.6"),
    ("joblib",           "Joblib",             "1.2"),
]

print(f"{'Package':<32} {'Status':<10} {'Version'}")
print("-" * 60)
missing = []
for import_name, display_name, min_ver in PACKAGES:
    try:
        mod = importlib.import_module(import_name)
        ver_str = getattr(mod, "__version__", "unknown")
        print(f"  ✔  {display_name:<28} {ver_str}")
    except ImportError:
        print(f"  ✘  {display_name:<28} NOT INSTALLED")
        missing.append(display_name)

print("\n" + "=" * 60)
if missing:
    print(f"  ⚠  {len(missing)} package(s) missing:")
    for pkg in missing:
        print(f"       - {pkg}")
    print("\n  Fix: pip install -r requirements.txt")
else:
    print("  ✔  All packages installed successfully!")
print("=" * 60)

In [ ]:
"""
Cell 4: GPU availability check (optional).
"""
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name  = torch.cuda.get_device_name(0)
        gpu_mem   = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✔  GPU detected  : {gpu_name}")
        print(f"   VRAM          : {gpu_mem:.1f} GB")
        print(f"   CUDA version  : {torch.version.cuda}")
    else:
        print("ℹ  No GPU detected — PyTorch will use CPU.")
        print("   BERT fine-tuning will be slow but functional.")
        print("   Alternative: Use Google Colab (free GPU) for BERT steps.")
    
    print(f"\n   PyTorch version: {torch.__version__}")
    device = "cuda" if gpu_available else "cpu"
    print(f"   Active device  : {device}")
except ImportError:
    print("⚠  PyTorch not installed. Run: pip install torch")

## Section 4 — Download NLP Resources

NLTK and spaCy require separate resource/model downloads that are **not** bundled with `pip install`.

| Resource | Purpose |
|----------|---------|
| `punkt` | Sentence tokenizer |
| `stopwords` | English stop word list |
| `wordnet` | Lemmatization dictionary |
| `averaged_perceptron_tagger` | Part-of-speech tagger |
| `omw-1.4` | Open Multilingual Wordnet |
| `en_core_web_sm` | spaCy English model (tokenize, lemmatize, NER) |

In [ ]:
"""
Cell 5: Download NLTK and spaCy resources.
Safe to re-run — already downloaded resources are skipped.
"""

import nltk
import subprocess

NLTK_RESOURCES = [
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "averaged_perceptron_tagger",
    "omw-1.4",
]

print("Downloading NLTK resources ...")
for resource in NLTK_RESOURCES:
    result = nltk.download(resource, quiet=True)
    status = "✔" if result else "already present"
    print(f"  {status}  nltk:{resource}")

print("\nChecking spaCy en_core_web_sm ...")
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    print(f"  ✔  spaCy model loaded: en_core_web_sm (v{nlp.meta['version']})")
except OSError:
    print("  ℹ  Downloading en_core_web_sm ...")
    result = subprocess.run(
        [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("  ✔  en_core_web_sm downloaded successfully")
    else:
        print(f"  ✘  Error: {result.stderr}")
        print("  Try manually: python -m spacy download en_core_web_sm")

print("\n✔ All NLP resources ready.")

## Section 5 — Reproducibility: Fix All Random Seeds

**Why this matters for research:**  
Without fixed seeds, every run produces different train/test splits, model initializations,  
and results — making your paper impossible to reproduce.

We fix seeds in: Python `random`, NumPy, PyTorch (CPU + GPU), and scikit-learn  
by passing `random_state=SEED` everywhere.

In [ ]:
"""
Cell 6: Fix all random seeds for full reproducibility.
Call set_seed() at the top of EVERY notebook.
"""

import random
import os
import numpy as np

def set_seed(seed: int = 42) -> None:
    """
    Fix random seeds across all libraries used in this project.
    
    Parameters
    ----------
    seed : int — master seed value (default: 42, from config)
    """
    # Python built-in
    random.seed(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # OS hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    # PyTorch (if available)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
        print(f"  ✔  PyTorch seed set (CPU + GPU)")
    except ImportError:
        pass

# ── Apply seed ────────────────────────────────────────────────
SEED = 42   # matches config.yaml project.seed

set_seed(SEED)

print(f"✔ Random seed fixed: {SEED}")
print(f"  - Python random   : seeded")
print(f"  - NumPy           : seeded")
print(f"  - PYTHONHASHSEED  : {os.environ['PYTHONHASHSEED']}")
print(f"\n  Important: Pass random_state={SEED} to ALL sklearn estimators!")

## Section 6 — Quick Smoke Test

We run a minimal end-to-end sanity check to confirm that the key libraries  
can be imported and used correctly:

- NumPy array operations
- Pandas DataFrame creation
- scikit-learn train/test split
- NLTK tokenization
- spaCy lemmatization
- VADER sentiment analysis

In [ ]:
"""
Cell 7: Smoke test — mini end-to-end pipeline check.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

SEED = 42
tests_passed = 0
tests_total  = 6

def run_test(name, fn):
    global tests_passed
    try:
        result = fn()
        print(f"  ✔  {name}")
        if result is not None:
            print(f"       → {result}")
        tests_passed += 1
    except Exception as e:
        print(f"  ✘  {name}")
        print(f"       Error: {e}")

print("Running smoke tests ...\n")

# ── Test 1: NumPy ────────────────────────────────────────────
def test_numpy():
    arr = np.array([1.5, 2.3, 4.1, 0.9, 3.7])
    return f"mean={arr.mean():.2f}, std={arr.std():.2f}"
run_test("NumPy array ops", test_numpy)

# ── Test 2: Pandas ───────────────────────────────────────────
def test_pandas():
    df = pd.DataFrame({
        "product_id":  ["P001", "P002", "P003"],
        "review_text": ["Great product!", "Terrible quality.", "Okay, I guess."],
        "rating":      [5, 1, 3],
        "is_fake":     [0, 1, 0],
    })
    return f"DataFrame shape: {df.shape}"
run_test("Pandas DataFrame", test_pandas)

# ── Test 3: scikit-learn split ───────────────────────────────
def test_sklearn():
    X = np.random.rand(100, 5)
    y = np.random.randint(0, 2, 100)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    return f"Train={len(X_train)}, Test={len(X_test)}"
run_test("Scikit-learn train/test split", test_sklearn)

# ── Test 4: NLTK tokenization ────────────────────────────────
def test_nltk():
    text  = "The product shipped quickly and worked perfectly!"
    stop  = set(stopwords.words("english"))
    tokens = [t.lower() for t in word_tokenize(text) if t.isalpha() and t.lower() not in stop]
    return f"Tokens: {tokens}"
run_test("NLTK tokenization + stopword removal", test_nltk)

# ── Test 5: spaCy lemmatization ──────────────────────────────
def test_spacy():
    import spacy
    nlp    = spacy.load("en_core_web_sm")
    doc    = nlp("The reviews were running suspiciously fast yesterday")
    lemmas = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    return f"Lemmas: {lemmas}"
run_test("spaCy lemmatization", test_spacy)

# ── Test 6: VADER sentiment ──────────────────────────────────
def test_vader():
    sia    = SentimentIntensityAnalyzer()
    scores = sia.polarity_scores("This product is absolutely amazing and fantastic!")
    return f"compound={scores['compound']:.3f}, pos={scores['pos']:.3f}"
run_test("VADER sentiment analysis", test_vader)

# ── Summary ───────────────────────────────────────────────────
print(f"\n{'='*40}")
print(f"  Smoke Tests: {tests_passed}/{tests_total} passed")
if tests_passed == tests_total:
    print("  ✔  Environment fully operational!")
else:
    print(f"  ⚠  {tests_total - tests_passed} test(s) failed.")
    print("     Fix the errors above before proceeding.")
print(f"{'='*40}")

## Section 7 — Project Health Report & Next Steps

After running all cells above, you should see all green checkmarks.  
This confirms the complete research environment is ready.

In [ ]:
"""
Cell 8: Final health report — summarizes all checks in one table.
"""

from datetime import datetime

print("=" * 65)
print("  MULTIMODAL FAKE PRODUCT HYPE DETECTION")
print("  Step 1 Health Report")
print(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 65)

checks = [
    ("Project root detected",      PROJECT_ROOT.exists()),
    ("config/config.yaml exists",  Path("config/config.yaml").exists()),
    ("data/raw/ exists",           Path("data/raw").exists()),
    ("models/saved/ exists",       Path("models/saved").exists()),
    ("reports/figures/ exists",    Path("reports/figures").exists()),
    ("logs/ exists",               Path("logs").exists()),
    ("src/utils/ exists",          Path("src/utils").exists()),
]

all_ok = True
for label, status in checks:
    icon = "✔" if status else "✘"
    if not status:
        all_ok = False
    print(f"  {icon}  {label}")

print("\n" + "-" * 65)

if all_ok:
    print("""
  ✔  All systems GO. Your research environment is ready.

  ┌─────────────────────────────────────────────────────────┐
  │                    NEXT STEPS                           │
  │                                                         │
  │  1. Place raw data files in data/raw/                   │
  │       - amazon_reviews.csv  (Amazon review dataset)     │
  │       - yelp_reviews.json   (Yelp review dataset)       │
  │                                                         │
  │  2. Say "NEXT" to proceed to:                           │
  │       Step 2 → Dataset Loading & Validation             │
  │       (notebooks/02_data_loading.ipynb)                 │
  └─────────────────────────────────────────────────────────┘
""")
else:
    print("""
  ⚠  Some checks failed. Review the ✘ items above and:
     1. Run: pip install -r requirements.txt
     2. Run: python scripts/setup_project.py
     3. Re-run this notebook
""")

print("=" * 65)